<a href="https://colab.research.google.com/github/zyf-hitsz/PytorchLearning/blob/main/%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C/%E6%8D%9F%E5%A4%B1%E5%87%BD%E6%95%B0/LossFunctions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#损失函数
计算真实值和模型预测结果之间的误差，用于衡量预测的效果如何，决定模型向哪个方向学习。

$$
前向传播得到预测
\rightarrow
Loss 衡量预测有多差
\rightarrow
反向传播计算梯度
\rightarrow
优化器更新参数
$$

真正重要的不是 Loss 本身，而是：
$$
\frac{\partial L}{\partial \theta}
$$梯度告诉模型：
参数应该往哪个方向改变，才能降低 Loss。

假设：
$$
\hat y=wx
$$使用：
$$
L=(\hat y-y)^2
$$那么：
$$
L=(wx-y)^2
$$对 \(w\) 求导：
$$
\frac{\partial L}{\partial w}
=
2(wx-y)x
$$梯度下降：
$$
w_{t+1}
=
w_t-\eta\frac{\partial L}{\partial w}
$$其中 $\eta$ 是学习率。
因此整个训练实际上是：
$$\theta^*=\arg\min_\theta L(f_\theta(x),y)$$
所以：Loss function 实际上定义了神经网络的优化目标。

换一个 Loss，即使网络结构完全一样，数据完全一样，优化器完全一样，最终训练出来的模型也可能完全不同。

##第一类：回归损失
回归问题预测的是连续值

###MSELoss：均方误差
MSE：
$$
\boxed{
L=(\hat y-y)^2
}
$$batch：
$$
L=
\frac1N
\sum_i
(\hat y_i-y_i)^2
$$
```python
class torch.nn.MSELoss(reduction='mean')
# reduction='mean'：
# non：不汇总，返回每个元素的平方误差。
# mean：返回所有元素平方误差的平均值，默认值。
# sum：返回所有元素平方误差之和。
```
特点：平方惩罚，误差越大，惩罚增长越快，梯度也越大。

缺点：对少数异常值特别敏感，比如一个异常的特别大的误差可能会主导整个batch的梯度。

In [1]:
import torch
import torch.nn as nn

pred = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.0, 4.0, 2.0])

# 每个元素的误差：(1-1)², (2-4)², (3-2)²
# 结果：[0, 4, 1]

print(nn.MSELoss(reduction="none")(pred, target))
# tensor([0., 4., 1.])

print(nn.MSELoss(reduction="mean")(pred, target))
# tensor(1.6667)

print(nn.MSELoss(reduction="sum")(pred, target))
# tensor(5.)

tensor([0., 4., 1.])
tensor(1.6667)
tensor(5.)


###L1Loss：平均绝对误差


```python
class torch.nn.L1Loss(reduction='mean')
```
L1比MSE对异常值更加鲁棒，但是梯度为±1，且在0处不可导。优化性质不平滑。



###Huber Loss：MSE和L1Loss的结合
数学定义：
$$
L_\delta(e)
=
\begin{cases}
\frac12e^2,& |e|\le\delta\\
\delta(|e|-\frac12\delta),& |e|>\delta
\end{cases}
$$
误差小时类似于MSE，误差大时类似于L1。兼顾MSE的平滑优化和L1的异常值鲁棒性。

```python
class torch.nn.HuberLoss(reduction='mean', delta=1.0)
```



In [4]:
# @title
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider


def huber_loss(e, delta):
    """计算 Huber 损失。"""
    abs_e = np.abs(e)

    return np.where(
        abs_e <= delta,
        0.5 * e**2,
        delta * (abs_e - 0.5 * delta)
    )


def plot_huber(delta=1.0):
    # e 的范围固定
    e = np.linspace(-5, 5, 1000)
    loss = huber_loss(e, delta)

    plt.figure(figsize=(8, 5))

    plt.plot(
        e,
        loss,
        linewidth=2.5,
        label=fr"$L_\delta(e),\ \delta={delta:.2f}$"
    )

    # 标出分界线 e = ±delta
    plt.axvline(-delta, color="red", linestyle="--", alpha=0.7)
    plt.axvline(delta, color="red", linestyle="--", alpha=0.7)

    # 标出两个分段点
    boundary_loss = 0.5 * delta**2
    plt.scatter(
        [-delta, delta],
        [boundary_loss, boundary_loss],
        color="red",
        zorder=3
    )

    plt.xlabel(r"Error $e$")
    plt.ylabel(r"Loss $L_\delta(e)$")
    plt.title("Huber Loss")
    plt.xlim(-5, 5)
    plt.ylim(0, 10)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


interact(
    plot_huber,
    delta=FloatSlider(
        value=1.0,
        min=0.1,
        max=4.0,
        step=0.1,
        description=r"δ"
    )
)

interactive(children=(FloatSlider(value=1.0, description='δ', max=4.0, min=0.1), Output()), _dom_classes=('wid…

<function __main__.plot_huber(delta=1.0)>

##第二类：分类损失
多分类、二分类、多标签分类



###CrossEntropyLoss
假设一个 $C$ 分类问题。
网络最后输出：
$$
z=
[z_1,z_2,\dots,z_C]
$$例如：
~~~
猫    2.3
狗   -0.7
鸟    1.1
~~~
这些不是概率。
它们叫：
$$
\boxed{\text{logits}}
$$然后 Softmax：
$$
p_i
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$得到概率。
假如：
$$
p=
[0.7,0.05,0.25]
$$真实类别是猫：
$$
y=0
$$那么 Cross Entropy：
$$
L=-\log p_y
$$也就是：
$$
L=-\log0.7
$$如果正确类别概率：
$$
p_y\rightarrow1
$$那么：
$$
L\rightarrow0
$$如果：
$$
p_y\rightarrow0
$$那么：
$$
L\rightarrow+\infty
$$这正符合我们需要的效果。

```python
class torch.nn.CrossEntropyLoss(weight=None,ignore_index=-100,reduction='mean', label_smoothing=0.0)
# weight=None:为每个类别设置损失权重，常用于不平衡问题
# ignore_index=-100：指定某个标签不参与损失计算，默认忽略标签-100
# label_smoothing=0.0：标签平滑系数，取值为0~1，默认0.0表示不适用标签平滑，标签平滑会把一部分概率分配给其他类别
```
nn.CrossEntropyLoss()
内部已经包含：
$$
\boxed{
\text{LogSoftmax}
+
\text{NLLLoss}
}
$$

####NLLLoss
即 Negative Log Likelihood：
$$
L=-\log p_y
$$但它要求输入是：
$$
\log p
$$
也就是：
$$
\boxed{
CrossEntropyLoss
=
LogSoftmax
+
NLLLoss
}
$$







###BCE（Binary Cross Entropy）和BCELoss
二分类中：
$$
y\in\{0,1\}
$$预测：
$$
p\in[0,1]
$$BCE：
$$
\boxed{
L=
-[y\log p+(1-y)\log(1-p)]
}
$$如果：
$$
y=1
$$那么：
$$
L=-\log p
$$如果：
$$
y=0
$$那么：
$$
L=-\log(1-p)
$$所以模型会：
正样本 → $p\to1$
负样本 → $p\to0$


```python
class torch.nn.BCELoss(weight=None, size_average=None, reduce=None, reduction='mean')
```
BECLoss要求输入已经是概率，即logits→Sigmoid→BCELoss

实际训练更推荐BCEWithLogitsLoss，因为它把：
$$
Sigmoid + BCE
$$合在了一起。不要再手动进行一次sigmoid()。

BCEWithLogitsLoss常用于多标签问题，不需要不同类别之间的概率之和为1，每个类别之间没有竞争，独立判断。



###KLDivLoss
KL divergence 衡量两个概率分布的差异。
对于：
$$
P(x), Q(x)
$$定义：
$$
\boxed{
D_{KL}(P\|Q)
=
\sum_x
P(x)
\log
\frac{P(x)}{Q(x)}
}
$$
```python
class torch.nn.KLDivLoss(size_average=None, reduce=None, reduction='mean', log_target=False)
```
KL divergence 不是严格意义上的距离，
因为一般：
$$D_{KL}(P\|Q)
\neq
D_{KL}(Q\|P)
$$它不是对称的。
同时：
$$
D_{KL}(P\|Q)\ge0
$$当：$P=Q$时：
$$D_{KL}=0$$
~~~
它经常出现在：
知识蒸馏
VAE
概率模型
分布匹配
teacher-student model
~~~

##第三类：Embedding / Metric Learning Loss
有时候我们不希望模型直接预测类别，而希望模型学到好的特征空间，比如人脸识别、图像检索、语义检索、对比学习等。我们希望：

相似样本 → embedding 接近

不相似样本 → embedding 远离

###CosineEmbeddingLoss
余弦相似度：
$$
\cos(x_1,x_2)
=
\frac{x_1^Tx_2}
{\|x_1\|\|x_2\|}
$$两个 embedding 相似：
$$
\cos\rightarrow1
$$不相似：
$$
\cos\rightarrow-1
$$
```python
class torch.nn.CosineEmbeddingLoss(margin=0.0, reduction='mean')
#margin=0.0:设置负样本的相似度边界。一般设置0-0.5之间。
#     margin 越小，对负样本的要求越严格。margin 越大，允许负样本具有更高的相似度。
```
设两个输入向量分别是$x_1$,$x_2$,标签为y，余弦相似度为$cos(x_1,x_2)$，
损失函数定义为：
$$L(x_1,x_2,y)=\begin{cases}
1-cos(x_1,x_2),& y=1\\
\max(0,cos(x_1,x_2)-margin),& y=-1
\end{cases}
$$

In [13]:
import torch
import torch.nn as nn

# 3对向量，每个向量有4个特征
x1 = torch.tensor([
    [1, 0.0, 0.1, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 0.9, 1.0, 0.1]
])

x2 = torch.tensor([
    [0.9, 0.1, 0.0, 0.0],
    [1.0, 0.8, 0.1, 0.0],
    [1.0, 0.0, 0.1, 0.0]
])

# 第一、二对应该相似，第三对应该不相似
target = torch.tensor([1, 1, -1])

loss_fn = nn.CosineEmbeddingLoss(
    margin=0.0,
    reduction="none"
)

loss = loss_fn(x1, x2, target)
print(loss)

tensor([0.0110, 0.0091, 0.0738])


###TripletMarginLoss
假设有三个样本：
$Anchor$,$Positive$,$Negative$

比如人脸识别：
```
Anchor：张三照片 A
Positive：张三照片 B
Negative：李四照片
```
希望：
$$
d(A,P)<d(A,N)
$$而且至少差一个 margin：
$$
d(A,P)+m<d(A,N)
$$Triplet loss：
$$
\boxed{
L=
\max
\left(
d(A,P)-d(A,N)+m,\ 0
\right)
}$$

```python
class torch.nn.TripletMarginLoss(margin=1.0, p=2.0, eps=1e-06, swap=False, reduction='mean')
#p=2.0：指定计算向量距离时使用的 p-范数，一般默认为p=2欧氏距离
#eps=1e-6：计算距离时加在分母上的数值稳定项
#swap=False：是否启用距离交换策略。
```



In [17]:
import torch
import torch.nn as nn

# 3个三元组，每个样本是4维特征向量
anchor = torch.randn(3, 4)
positive = torch.randn(3, 4)
negative = torch.randn(3, 4)

loss_fn = nn.TripletMarginLoss(
    margin=1.0,
    p=2.0,
    eps=1e-6,
    swap=False,
    reduction="mean"
)

loss = loss_fn(anchor, positive, negative)

print(loss)

tensor(1.1804)


###多Loss联合训练
很多现代模型真正的 objective 是复合Loss：
$$
\boxed{
L_{\text{total}}
=
\sum_i\lambda_iL_i
}
$$
Loss小不一定训练效果就好，更重要的是准确率accuracy